In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import SGDClassifier
import numpy as np
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from mlflow.models import infer_signature
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

/home/alina/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocessing_data_frame(frame):
    df = frame.copy()
    cat_columns = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
    num_columns = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']
    
    # Анализ и очистка данных
    
    # здравый смысл
    question_engine = df[(df["RestingBP"] < 30) | (df["RestingBP"] > 250)] #уровень артериального давления у человека в сознании не м.б < 50  и > 250
    df = df.drop(question_engine.index)
    
    # здравый смысл
    question_engine = df[(df["Cholesterol"] == 0) | (df["Cholesterol"] > 600)] # то же для холестирина
    df = df.drop(question_engine.index) 
    
    # здравый смысл
    question_price = df[(df["Age"] <= 0) | (df["Age"] > 120)]
    df = df.drop(question_price.index)

    # здравый смысл
    question_price = df[~df['Sex'].isin(['M', 'F'])]
    df = df.drop(question_price.index)  
    
    df = df.reset_index(drop=True)  # обновим индексы в датафрейме DF. если бы мы прописали drop = False, то была бы еще одна колонка - старые индексы
    # Разделение данных на признаки и целевую переменную
    
    
    # Предварительная обработка категориальных данных
    # Порядковое кодирование. Обучение, трансформация и упаковка в df
    
    ordinal = OrdinalEncoder()
    ordinal.fit(df[cat_columns]);
    Ordinal_encoded = ordinal.transform(df[cat_columns])
    df_ordinal = pd.DataFrame(Ordinal_encoded, columns=cat_columns)
    df[cat_columns] = df_ordinal[cat_columns]
    return df

def scale_frame(frame):
    df = frame.copy()
    X,y = df.drop(columns = ['HeartDisease']), df['HeartDisease']
    scaler = StandardScaler()
    X_scale = scaler.fit_transform(X.values)
    return X_scale, y, scaler

In [3]:
df = pd.read_csv('heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:
df_proc = preprocessing_data_frame(df)
X, y, scaler = scale_frame(df_proc)
# разбиваем на тестовую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, y,
                                                  test_size=0.3,
                                                  random_state=42)

In [5]:
def eval_metrics(actual, pred):
    accuracy = accuracy_score(actual, pred)
    precision = precision_score(actual, pred)
    recall = recall_score(actual, pred)
    f1 = f1_score(actual, pred)
    return accuracy, precision, recall, f1

In [ ]:
params = {
    'alpha': [0.0008, 0.0009, 0.001, 0.002],
    'l1_ratio': [0.1, 0.3, 0.5, 0.8, 0.9],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'loss': ['log_loss', 'hinge'],
    'fit_intercept': [True, False],
    'eta0': [0.00001, 0.0001, 0.001, 0.01, 0.1],
    'learning_rate': ['optimal', 'invscaling', 'adaptive']
}  
with mlflow.start_run():
    lr = SGDClassifier(random_state=42)
    clf = RandomizedSearchCV(lr, params, cv=3, n_jobs=4, n_iter=10, random_state=None)
    clf.fit(X_train, y_train)
    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    (accuracy, precision, recall, f1)  = eval_metrics(y_val, y_pred)
    alpha = best.alpha
    l1_ratio = best.l1_ratio
    mlflow.log_param("alpha", best.alpha)
    mlflow.log_param("l1_ratio", best.l1_ratio)
    mlflow.log_param("penalty", best.penalty)
    mlflow.log_param("loss", best.loss)
    mlflow.log_param("fit_intercept", best.fit_intercept)
    mlflow.log_param("eta0", best.eta0)
    mlflow.log_param("learning_rate", best.learning_rate)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    
    predictions = best.predict(X_train)
    signature = infer_signature(X_train, predictions)
    mlflow.sklearn.log_model(best, "model", signature=signature)

2026/04/15 03:29:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 03:29:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run unique-goat-837 at: http://127.0.0.1:5000/#/experiments/0/runs/bd4bfd99c30a4e2dbf3175bf23f201d6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


In [19]:
best.predict(X_val[122:123])[0]

np.int64(1)

In [20]:
y_val.iloc[122]

np.int64(1)

In [22]:
!ls ./mlartifacts/0/

models


In [118]:
## История запуска моделей

In [23]:
dfruns = mlflow.search_runs()
frame = dfruns.sort_values("metrics.accuracy", ascending=False)
frame

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.recall,metrics.accuracy,metrics.precision,metrics.f1,...,params.eta0,params.l1_ratio,params.fit_intercept,params.loss,params.penalty,params.learning_rate,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.source.type
0,cbfe4e52c4e74347bc7eb74289ba8878,0,FINISHED,mlflow-artifacts:/0/cbfe4e52c4e74347bc7eb74289...,2026-04-08 11:16:11.626000+00:00,2026-04-08 11:16:30.887000+00:00,0.798246,0.84375,0.883495,0.83871,...,0.0001,0.9,True,hinge,elasticnet,optimal,auspicious-goat-212,alina,mlflow.ipynb,NOTEBOOK
1,5dcd664b9ccf417f88cff16ad1dbdd50,0,FAILED,mlflow-artifacts:/0/5dcd664b9ccf417f88cff16ad1...,2026-04-08 11:13:04.040000+00:00,2026-04-08 11:13:24.794000+00:00,0.798246,0.84375,0.883495,0.83871,...,0.0001,0.9,True,hinge,elasticnet,optimal,respected-carp-186,alina,mlflow.ipynb,NOTEBOOK
2,a37a396305ac40b195a03c9457c19cc5,0,FAILED,mlflow-artifacts:/0/a37a396305ac40b195a03c9457...,2026-04-08 11:09:24.979000+00:00,2026-04-08 11:09:47.049000+00:00,0.798246,0.84375,0.883495,0.83871,...,0.0001,0.9,True,hinge,elasticnet,optimal,resilient-bird-578,alina,mlflow.ipynb,NOTEBOOK


In [24]:
dfruns

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.recall,metrics.accuracy,metrics.precision,metrics.f1,...,params.eta0,params.l1_ratio,params.fit_intercept,params.loss,params.penalty,params.learning_rate,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.source.type
0,cbfe4e52c4e74347bc7eb74289ba8878,0,FINISHED,mlflow-artifacts:/0/cbfe4e52c4e74347bc7eb74289...,2026-04-08 11:16:11.626000+00:00,2026-04-08 11:16:30.887000+00:00,0.798246,0.84375,0.883495,0.83871,...,0.0001,0.9,True,hinge,elasticnet,optimal,auspicious-goat-212,alina,mlflow.ipynb,NOTEBOOK
1,5dcd664b9ccf417f88cff16ad1dbdd50,0,FAILED,mlflow-artifacts:/0/5dcd664b9ccf417f88cff16ad1...,2026-04-08 11:13:04.040000+00:00,2026-04-08 11:13:24.794000+00:00,0.798246,0.84375,0.883495,0.83871,...,0.0001,0.9,True,hinge,elasticnet,optimal,respected-carp-186,alina,mlflow.ipynb,NOTEBOOK
2,a37a396305ac40b195a03c9457c19cc5,0,FAILED,mlflow-artifacts:/0/a37a396305ac40b195a03c9457...,2026-04-08 11:09:24.979000+00:00,2026-04-08 11:09:47.049000+00:00,0.798246,0.84375,0.883495,0.83871,...,0.0001,0.9,True,hinge,elasticnet,optimal,resilient-bird-578,alina,mlflow.ipynb,NOTEBOOK


In [25]:
model_uri = dfruns.sort_values("metrics.recall", ascending=False).iloc[0]['artifact_uri']
path2model = dfruns.sort_values("metrics.recall", ascending=False).iloc[0]['artifact_uri'].replace("file://","") + '/model' #путь до эксперимента с лучшей моделью
print(path2model)

mlflow-artifacts:/0/cbfe4e52c4e74347bc7eb74289ba8878/artifacts/model


In [26]:
loaded_model = mlflow.sklearn.load_model(path2model)
loaded_model

KeyboardInterrupt: 

In [ ]:
loaded_model = mlflow.sklearn.load_model(path2model)

test_input = np.array([[0.5, -1.0, 1.2, 0.1,-0.5, 0.0, 1.0, -0.8, 1.0, 0.5, -1.0]])
prediction = loaded_model.predict(test_input)
prediction

array([0.71926926])

### Запускает модель в сервис локально 

In [124]:
!mlflow models serve -m /media/kirilman/Z3/URFU/MLOPS/Лабы/MLOPS/mlflow/mlruns/0/d9f5c7d863d24d1591830bb7b15fd01b/artifacts/model/ -p 5001 --env-manager local

2026/03/25 12:33:43 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/03/25 12:33:43 INFO mlflow.pyfunc.backend: === Running command 'exec gunicorn --timeout=60 -b 127.0.0.1:5001 -w 1 ${GUNICORN_CMD_ARGS} -- mlflow.pyfunc.scoring_server.wsgi:app'
[2026-03-25 12:33:43 +0500] [303745] [INFO] Starting gunicorn 21.2.0
[2026-03-25 12:33:43 +0500] [303745] [INFO] Listening at: http://127.0.0.1:5001 (303745)
[2026-03-25 12:33:43 +0500] [303745] [INFO] Using worker: sync
[2026-03-25 12:33:43 +0500] [303746] [INFO] Booting worker with pid: 303746
[2026-03-25 12:34:56 +0500] [303745] [CRITICAL] WORKER TIMEOUT (pid:303746)
[2026-03-25 12:34:56 +0500] [303746] [INFO] Worker exiting (pid: 303746)
[2026-03-25 12:34:57 +0500] [303745] [ERROR] Worker (pid:303746) exited with code 1
[2026-03-25 12:34:57 +0500] [303745] [ERROR] Worker (pid:303746) exited with code 1.
[2026-03-25 12:34:57 +0500] [304103] [INFO] Booting worker with pid: 304103
^C
[2026-03-25 15

### Тестовый запрос к модели через curl

In [19]:
!curl http://127.0.0.1:5001/invocations -H"Content-Type:application/json"  --data '{"inputs": [[ 0.08265284, -0.18711058,  0.51492644, -1.41445941,  0.28915819, 0.54708212, -1.01395689, -1.09462628 ]]}'

mlflow.ipynb  mlruns
